# Post-Training: See Every Stage Transform a Model

This notebook has one goal: **watch a raw language model become a helpful assistant.**

We run every stage, generate text at each checkpoint, and see exactly what changed.

```
Stage 0: BASE MODEL        → completes text like autocomplete, can't follow instructions
         "What is gravity?" → "What is gravity? What is the meaning of gravity? In..."
              |
Stage 1: SFT (fine-tune)   → learns instruction → response format
         "What is gravity?" → "Gravity is a fundamental force that attracts..."
              |
Stage 2: DPO (align)       → learns WHICH answers humans prefer
         "What is gravity?" → "Gravity is the force of attraction between objects
                               with mass. It keeps planets in orbit and causes
                               objects to fall to the ground..."
```

**Runtime:** T4 GPU (free on Kaggle or Colab) — takes ~10-15 minutes total

## Step 1: Install & Setup

In [ ]:
!pip install -q transformers trl datasets peft accelerate

In [ ]:
import torch, os, time, gc
os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "Need a GPU!"
print("GPU:", torch.cuda.get_device_name(0))
print("Memory: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

MODEL = "Qwen/Qwen2.5-0.5B"  # small enough to run fast, big enough to show real behavior
print("Model:", MODEL)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Test prompts we'll use at every stage to see the transformation
TEST_PROMPTS = [
    "What is gravity?",
    "How do I make scrambled eggs?",
    "Explain Python lists to a beginner.",
    "Why is the sky blue?",
    "Is it true that we only use 10% of our brain?",
]

def generate(model, prompt, max_tokens=150):
    """Generate a response from a model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def generate_raw(model, prompt, max_tokens=100):
    """Generate without chat template — shows raw completion behavior."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Store all responses for final comparison
all_responses = {}

print("Ready. We'll test with %d prompts at each stage." % len(TEST_PROMPTS))

---
## Stage 0: The Base Model (Before Post-Training)

A pre-trained model is like a student who has **read millions of books**
but has **never had a conversation**.

It knows facts, grammar, and patterns — but if you ask it a question,
it just continues the text like autocomplete.

```
You type:    "What is gravity?"
It thinks:   "This looks like the start of a paragraph or essay..."
It outputs:  "What is gravity? What is the force of gravity? How does
              gravity work? These are questions that have puzzled..."

It doesn't ANSWER. It CONTINUES.
```

In [ ]:
# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True
).to("cuda")
base_model.eval()

print("=" * 70)
print("  STAGE 0: BASE MODEL (pre-trained, no post-training)")
print("  The model just CONTINUES text. It doesn't know how to ANSWER.")
print("=" * 70)

all_responses["base"] = {}

for prompt in TEST_PROMPTS:
    # Show RAW completion (no chat template)
    raw_response = generate_raw(base_model, prompt, max_tokens=80)
    # Also try with chat template
    chat_response = generate(base_model, prompt, max_tokens=80)
    
    all_responses["base"][prompt] = chat_response
    
    print("\nQ: %s" % prompt)
    print("  [raw completion]:  %s" % raw_response[:200])
    print("  [chat template]:   %s" % chat_response[:200])
    print("-" * 70)

print("\nNotice: The base model often repeats the question, rambles,")
print("or generates unrelated text. It hasn't learned to be an assistant yet.")

del base_model
gc.collect()
torch.cuda.empty_cache()

---
## Stage 1: SFT (Supervised Fine-Tuning)

### What SFT does

We show the model thousands of examples of:
```
User: <question>
Assistant: <good answer>
```

The model learns: when someone asks a question, **generate an answer** (don't just continue).

### How it learns

SFT uses the same loss as pre-training: **next-token prediction**.
But instead of predicting the next word in a book, it predicts
the next word in an instruction-response pair.

```
Training example:
  Input:  "User: What is gravity? Assistant:"
  Target: "Gravity is a fundamental force..."
  Loss:   cross-entropy on each token of the answer

After thousands of examples, the model learns:
  1. The format: User asks → Assistant answers
  2. The style: be helpful, clear, concise
  3. The content: use its knowledge to answer
```

### What SFT does NOT do

SFT teaches the model to give **an** answer.
It does NOT teach it to give the **best** answer.
That's what Stage 2 (DPO) is for.

In [ ]:
from datasets import load_dataset

# Load instruction-tuning data
sft_data = load_dataset("tatsu-lab/alpaca", split="train")
sft_data = sft_data.shuffle(seed=42).select(range(2000))

def format_sft(example):
    prompt = example["instruction"]
    if example.get("input"):
        prompt += "\n" + example["input"]
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

sft_formatted = sft_data.map(format_sft, remove_columns=sft_data.column_names)

print("SFT dataset: %d instruction-response pairs" % len(sft_formatted))
print("\nExample:")
print(sft_formatted[0]["text"][:300])

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

# Load fresh model for SFT
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True
).to("cuda")

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=SFTConfig(
        output_dir="./sft_output",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=20,
        logging_steps=10,
        bf16=True,
        gradient_checkpointing=True,
        max_length=512,
        dataset_text_field="text",
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=sft_formatted,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Starting SFT...")
sft_start = time.time()
sft_trainer.train()
sft_time = time.time() - sft_start
print("SFT done in %.0f seconds!" % sft_time)

# Save SFT checkpoint (we'll load it for DPO)
sft_trainer.save_model("./sft_output/final")
tokenizer.save_pretrained("./sft_output/final")

In [ ]:
# Test SFT model
sft_model.eval()

print("=" * 70)
print("  STAGE 1: AFTER SFT")
print("  The model now ANSWERS questions instead of continuing text.")
print("=" * 70)

all_responses["sft"] = {}

for prompt in TEST_PROMPTS:
    response = generate(sft_model, prompt)
    all_responses["sft"][prompt] = response
    print("\nQ: %s" % prompt)
    print("A: %s" % response[:300])
    print("-" * 70)

print("\nNotice: The model now gives real answers! But they might be")
print("short, generic, or not the BEST possible answer. That's next.")

del sft_trainer
gc.collect()
torch.cuda.empty_cache()

---
## Stage 2: DPO (Direct Preference Optimization)

### What DPO does

SFT taught the model to answer. DPO teaches it which answers are **better**.

```
Prompt: "What is gravity?"

Chosen (good):    "Gravity is the force of attraction between objects
                   with mass. It's what keeps planets in orbit and
                   causes objects to fall to the ground."              <- detailed, accurate

Rejected (bad):   "Gravity is a thing. It makes stuff fall."          <- lazy, vague

DPO loss: make P(chosen) > P(rejected)
```

### How DPO works (the core idea)

```
For each preference pair (prompt, chosen, rejected):

  1. Compute log-probability of chosen response under our model
  2. Compute log-probability of rejected response under our model
  3. Compute same log-probs under the reference model (frozen SFT model)
  4. Push our model to increase the gap:
     
     loss = -log(sigmoid(β * [(log P_model(chosen) - log P_ref(chosen))
                             - (log P_model(rejected) - log P_ref(rejected))]))
     
  The β parameter (0.1) controls how aggressively the model changes.
  The reference model prevents the model from changing too much.
```

### Why DPO over RLHF?

```
RLHF (old):  Train reward model → Run PPO → unstable, complex
DPO  (new):  Train directly on preference pairs → stable, simple

Same result, 1/3 the code, way easier to train.
```

In [ ]:
from datasets import Dataset

# Preference pairs: chosen (good) vs rejected (bad) responses
# These teach the model QUALITY differences
preference_pairs = [
    {
        "prompt": "What is gravity?",
        "chosen": "Gravity is the fundamental force of attraction between objects with mass. It's described by Newton's law of universal gravitation and Einstein's general relativity. In everyday life, gravity keeps us on the ground, makes objects fall, and keeps planets orbiting stars.",
        "rejected": "Gravity is a force. It makes things fall down."
    },
    {
        "prompt": "How do I make scrambled eggs?",
        "chosen": "Crack 2-3 eggs into a bowl and whisk with a pinch of salt and pepper. Heat a non-stick pan over medium-low heat with a tablespoon of butter. Pour in the eggs and gently stir with a spatula, pushing curds from the edges toward the center. Remove from heat when slightly underdone — they'll continue cooking. Serve immediately.",
        "rejected": "Put eggs in a pan and cook them."
    },
    {
        "prompt": "Explain Python lists to a beginner.",
        "chosen": "A Python list is an ordered collection that can hold multiple items. Create one with square brackets: `colors = ['red', 'blue', 'green']`. Access items by position: `colors[0]` gives 'red'. Add items with `colors.append('yellow')`. Lists can hold any type — numbers, strings, even other lists. They're one of the most useful tools in Python.",
        "rejected": "Lists are data structures in Python. They store elements."
    },
    {
        "prompt": "Why is the sky blue?",
        "chosen": "The sky appears blue because of Rayleigh scattering. When sunlight enters Earth's atmosphere, it collides with gas molecules. Blue light has a shorter wavelength, so it scatters more than other colors in all directions. When you look up, you see this scattered blue light coming from everywhere in the sky.",
        "rejected": "The sky is blue because of the atmosphere."
    },
    {
        "prompt": "Is it true that we only use 10% of our brain?",
        "chosen": "No, this is a myth. Brain imaging studies show that we use virtually all parts of our brain, and most of the brain is active most of the time. Different areas handle different functions — motor control, vision, language, memory. Even during sleep, areas like the frontal cortex and somatosensory areas are active.",
        "rejected": "Yes, we only use 10% of our brain. The rest is unused potential."
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed. For example, a spam filter learns from thousands of labeled emails what spam looks like. The three main types are supervised learning (labeled data), unsupervised learning (finding patterns), and reinforcement learning (learning from rewards).",
        "rejected": "Machine learning is AI stuff. Computers learn things."
    },
    {
        "prompt": "How does the internet work?",
        "chosen": "When you visit a website, your browser sends a request through your ISP to a DNS server, which translates the domain name to an IP address. The request travels through routers to the web server, which sends back the page data in packets. These packets may take different routes but are reassembled by your browser into the page you see. This all happens using the TCP/IP protocol stack.",
        "rejected": "The internet uses wifi and cables to connect computers."
    },
    {
        "prompt": "What is photosynthesis?",
        "chosen": "Photosynthesis is how plants convert light energy into chemical energy (food). In the chloroplasts of plant cells, chlorophyll absorbs sunlight and uses it to convert carbon dioxide from the air and water from the soil into glucose and oxygen. The equation: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2.",
        "rejected": "Plants use sunlight to make food somehow."
    },
    {
        "prompt": "Explain recursion in programming.",
        "chosen": "Recursion is when a function calls itself to solve a problem by breaking it into smaller sub-problems. Each call handles a smaller piece until reaching a base case that stops the recursion. Example: factorial(5) = 5 × factorial(4) = 5 × 4 × factorial(3) = ... = 5 × 4 × 3 × 2 × 1 = 120. Every recursive function needs a base case (factorial(1) = 1) to avoid infinite loops.",
        "rejected": "Recursion is when a function calls itself. It's complicated."
    },
    {
        "prompt": "Why do we dream?",
        "chosen": "Scientists don't fully agree, but leading theories include: memory consolidation (the brain processes and stores important information from the day), emotional regulation (dreams help process difficult emotions), and threat simulation (practicing responses to dangers). During REM sleep, the brain is highly active while the body is paralyzed, creating vivid dream experiences.",
        "rejected": "Nobody really knows why we dream."
    },
]

def format_dpo(pair):
    user_msg = {"role": "user", "content": pair["prompt"]}
    return {
        "prompt": [user_msg],
        "chosen": [user_msg, {"role": "assistant", "content": pair["chosen"]}],
        "rejected": [user_msg, {"role": "assistant", "content": pair["rejected"]}],
    }

dpo_dataset = Dataset.from_list([format_dpo(p) for p in preference_pairs])

print("DPO dataset: %d preference pairs" % len(dpo_dataset))
print("\nExample pair:")
print("  Prompt:   %s" % preference_pairs[0]["prompt"])
print("  Chosen:   %s" % preference_pairs[0]["chosen"][:80])
print("  Rejected: %s" % preference_pairs[0]["rejected"][:80])

In [ ]:
from trl import DPOConfig, DPOTrainer

# DPO trains on top of the SFT model
dpo_trainer = DPOTrainer(
    model=sft_model,  # start from SFT checkpoint
    args=DPOConfig(
        output_dir="./dpo_output",
        beta=0.1,                        # controls how much to trust preferences
        num_train_epochs=3,              # 3 epochs over 10 pairs
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,              # lower than SFT — gentle alignment
        warmup_steps=5,
        logging_steps=1,                 # log every step to see DPO dynamics
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        max_length=512,
        max_prompt_length=128,
        remove_unused_columns=False,
    ),
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("Starting DPO...")
print("  beta=0.1 — moderate preference strength")
print("  lr=5e-5 — gentle updates (5x lower than SFT)")

dpo_start = time.time()
dpo_trainer.train()
dpo_time = time.time() - dpo_start
print("DPO done in %.0f seconds!" % dpo_time)

# Save DPO model
dpo_trainer.save_model("./dpo_output/final")
tokenizer.save_pretrained("./dpo_output/final")

In [ ]:
# Test DPO model
dpo_model = dpo_trainer.model
dpo_model.eval()

print("=" * 70)
print("  STAGE 2: AFTER DPO")
print("  The model now gives BETTER, more detailed answers.")
print("=" * 70)

all_responses["dpo"] = {}

for prompt in TEST_PROMPTS:
    response = generate(dpo_model, prompt)
    all_responses["dpo"][prompt] = response
    print("\nQ: %s" % prompt)
    print("A: %s" % response[:300])
    print("-" * 70)

del dpo_trainer
gc.collect()
torch.cuda.empty_cache()

---
## The Full Transformation: Side by Side

In [ ]:
from IPython.display import HTML, display

# Build comparison table
rows = ""
for prompt in TEST_PROMPTS:
    base_r = all_responses["base"].get(prompt, "N/A")[:200]
    sft_r = all_responses["sft"].get(prompt, "N/A")[:200]
    dpo_r = all_responses["dpo"].get(prompt, "N/A")[:200]
    
    rows += """
    <tr style="border-bottom:1px solid #21262d;">
      <td style="padding:12px;color:#a78bfa;font-weight:600;vertical-align:top;min-width:120px;">%s</td>
      <td style="padding:12px;color:#f87171;vertical-align:top;font-size:12px;">%s</td>
      <td style="padding:12px;color:#fbbf24;vertical-align:top;font-size:12px;">%s</td>
      <td style="padding:12px;color:#3fb950;vertical-align:top;font-size:12px;">%s</td>
    </tr>
    """ % (prompt, base_r, sft_r, dpo_r)

html = """
<div style="font-family:-apple-system,sans-serif;margin:20px 0;overflow-x:auto;">
  <h3 style="color:#c9d1d9;">The Full Transformation</h3>
  <table style="width:100%%;border-collapse:collapse;color:#c9d1d9;background:#0d1117;">
    <tr style="border-bottom:2px solid #30363d;">
      <th style="padding:12px;text-align:left;color:#8b949e;">Prompt</th>
      <th style="padding:12px;text-align:left;color:#f87171;">Stage 0: Base<br><span style="font-weight:400;font-size:11px;">just autocompletes</span></th>
      <th style="padding:12px;text-align:left;color:#fbbf24;">Stage 1: SFT<br><span style="font-weight:400;font-size:11px;">answers questions</span></th>
      <th style="padding:12px;text-align:left;color:#3fb950;">Stage 2: DPO<br><span style="font-weight:400;font-size:11px;">gives better answers</span></th>
    </tr>
    %s
  </table>
</div>
""" % rows

display(HTML(html))

In [ ]:
# Measure response quality metrics
import numpy as np

metrics = {}
for stage in ["base", "sft", "dpo"]:
    lengths = [len(all_responses[stage][p].split()) for p in TEST_PROMPTS]
    metrics[stage] = {
        "avg_words": np.mean(lengths),
        "min_words": min(lengths),
        "max_words": max(lengths),
    }

print("Response length comparison:")
print("  %-10s  Avg words  Min  Max" % "Stage")
print("  " + "-" * 40)
for stage, label in [("base", "Base"), ("sft", "After SFT"), ("dpo", "After DPO")]:
    m = metrics[stage]
    print("  %-10s  %-9.0f  %-4.0f %-4.0f" % (label, m["avg_words"], m["min_words"], m["max_words"]))

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Average response length per stage
stages = ["Base", "After SFT", "After DPO"]
avg_lengths = [metrics[s]["avg_words"] for s in ["base", "sft", "dpo"]]
colors = ["#f87171", "#fbbf24", "#3fb950"]

bars = axes[0].bar(stages, avg_lengths, color=colors, edgecolor="white", linewidth=0.5)
axes[0].set_title("Average Response Length (words)", fontsize=14)
axes[0].set_ylabel("Words")
for bar, val in zip(bars, avg_lengths):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 "%.0f" % val, ha="center", fontsize=12, fontweight="bold")

# Chart 2: Response length per prompt per stage
x = np.arange(len(TEST_PROMPTS))
w = 0.25
short_prompts = [p[:20] + "..." for p in TEST_PROMPTS]

for i, (stage, color, label) in enumerate(zip(["base", "sft", "dpo"], colors, stages)):
    lengths = [len(all_responses[stage][p].split()) for p in TEST_PROMPTS]
    axes[1].bar(x + i * w, lengths, w, color=color, label=label, edgecolor="white", linewidth=0.5)

axes[1].set_xticks(x + w)
axes[1].set_xticklabels(short_prompts, rotation=25, ha="right", fontsize=8)
axes[1].set_title("Response Length per Question", fontsize=14)
axes[1].set_ylabel("Words")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

In [ ]:
from IPython.display import HTML, display

html = """
<div style="font-family:-apple-system,sans-serif;max-width:820px;margin:20px 0;">

  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:16px;">
    <div style="background:linear-gradient(135deg,#2d1b1b,#161b22);border:1px solid #f8717155;
                border-radius:12px;padding:20px;text-align:center;">
      <div style="color:#f87171;font-size:14px;font-weight:700;">Stage 0: Base</div>
      <div style="color:#8b949e;font-size:12px;margin-top:8px;">Pre-trained only</div>
      <div style="color:#8b949e;font-size:11px;margin-top:4px;">Autocompletes text</div>
      <div style="color:#8b949e;font-size:11px;">Can't follow instructions</div>
      <div style="color:#f87171;font-size:20px;font-weight:700;margin-top:8px;">%.0f words avg</div>
    </div>
    <div style="background:linear-gradient(135deg,#2d2b1b,#161b22);border:1px solid #fbbf2455;
                border-radius:12px;padding:20px;text-align:center;">
      <div style="color:#fbbf24;font-size:14px;font-weight:700;">Stage 1: SFT</div>
      <div style="color:#8b949e;font-size:12px;margin-top:8px;">%.0fs training</div>
      <div style="color:#8b949e;font-size:11px;margin-top:4px;">Learns to answer</div>
      <div style="color:#8b949e;font-size:11px;">Instruction → Response</div>
      <div style="color:#fbbf24;font-size:20px;font-weight:700;margin-top:8px;">%.0f words avg</div>
    </div>
    <div style="background:linear-gradient(135deg,#1b2d1e,#161b22);border:1px solid #3fb95055;
                border-radius:12px;padding:20px;text-align:center;">
      <div style="color:#3fb950;font-size:14px;font-weight:700;">Stage 2: DPO</div>
      <div style="color:#8b949e;font-size:12px;margin-top:8px;">%.0fs training</div>
      <div style="color:#8b949e;font-size:11px;margin-top:4px;">Learns quality</div>
      <div style="color:#8b949e;font-size:11px;">Prefers better answers</div>
      <div style="color:#3fb950;font-size:20px;font-weight:700;margin-top:8px;">%.0f words avg</div>
    </div>
  </div>

  <div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:18px;">
    <table style="width:100%%;color:#c9d1d9;font-size:13px;border-spacing:0 8px;">
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right;font-weight:600;">Qwen2.5-0.5B</td></tr>
      <tr><td style="color:#8b949e;">SFT dataset</td><td style="text-align:right;">2,000 Alpaca instruction pairs</td></tr>
      <tr><td style="color:#8b949e;">DPO dataset</td><td style="text-align:right;">10 preference pairs (chosen vs rejected)</td></tr>
      <tr><td style="color:#8b949e;">Method</td><td style="text-align:right;">LoRA (r=32, all linear layers)</td></tr>
      <tr><td style="color:#8b949e;">Total time</td><td style="text-align:right;font-weight:600;">%.0f seconds</td></tr>
    </table>
  </div>
</div>
""" % (
    metrics["base"]["avg_words"],
    sft_time, metrics["sft"]["avg_words"],
    dpo_time, metrics["dpo"]["avg_words"],
    sft_time + dpo_time,
)

display(HTML(html))

---
## The Core Ideas (What You Just Saw)

### 1. Pre-training ≠ Post-training

```
Pre-training:   "Read the entire internet"     → knows facts, grammar, patterns
Post-training:  "Learn to be an assistant"      → follows instructions, gives good answers

Pre-training costs: millions of dollars, months of compute
Post-training costs: a few dollars, hours on one GPU

Most of the "intelligence" comes from pre-training.
Post-training just teaches the model HOW to use that intelligence.
```

### 2. SFT teaches FORMAT, DPO teaches QUALITY

```
SFT:  "When someone asks X, respond with Y"     → learns the pattern
DPO:  "This answer is better than that answer"   → learns preferences

SFT makes the model helpful.
DPO makes the model better at being helpful.
```

### 3. You don't need millions of examples

```
SFT:  2,000 examples was enough to teach instruction-following
DPO:  10 preference pairs was enough to shift response quality

Why? The model already KNOWS the information from pre-training.
Post-training just teaches it how to present that knowledge.
```

### 4. The post-training pipeline in production

```
What OpenAI/Anthropic/Google actually do:

  1. Pre-train on trillions of tokens          (months, $100M+)
  2. SFT on ~100K high-quality conversations    (days, $10K)
  3. RLHF/DPO on human preference data          (days, $10K)
  4. Safety training (red-teaming, filtering)    (weeks)
  5. Repeat steps 2-4 with better data           (continuous)

What you just did in this notebook:

  1. Used a pre-trained model (Qwen2.5-0.5B)    (downloaded)
  2. SFT on 2,000 Alpaca examples               (minutes)
  3. DPO on 10 preference pairs                  (seconds)

Same pipeline. Smaller scale. Same core ideas.
```